# Lotka–Volterra: Recovery Probability, Conditioning, Coherence, and σ_min

This notebook applies the **same recovery-probability framework** used for the Lorenz system
to the Lotka–Volterra predator–prey model. For each polynomial degree and noise level it runs
many random seeds and records:

- recovery probability (fraction of seeds with correct sparse support)
- mean/std condition number of the library
- mean/std mutual coherence
- mean/std σ_min (smallest singular value of the column-normalised library)
- numerical rank and rank deficiency

True system (parameters from the project's Lotka example):

$$\dot{x} = \alpha x - \beta x y, \qquad \dot{y} = \delta x y - \gamma y$$

with α=1.0, β=0.1, δ=0.05, γ=1.5 — so in a degree-2 polynomial library the true support is
{`x`, `xy`} for ẋ and {`y`, `xy`} for ẏ. Note the coefficients (0.05, 0.1) are an order of
magnitude smaller than Lorenz's, which makes thresholding and noise more delicate here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from scipy.integrate import solve_ivp
from itertools import combinations_with_replacement

In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================

ALPHA = 1.0
BETA  = 0.1
DELTA = 0.05
GAMMA = 1.5

X0 = [40.0, 9.0]

DT = 0.01
TSPAN = (0, 50)

DEFAULT_THRESHOLD = 0.025  # see note below: must sit BELOW the smallest
                           # true coefficient (delta = 0.05), not on top of it

DEGREES = [1, 2, 3, 4, 5]

NOISE_LEVELS = [
    0,
    0.001,
    0.005,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

> **Threshold choice (important Lotka-specific subtlety).** The smallest true coefficient in
> this system is δ = 0.05 (the `xy` term in ẏ). If the STLS threshold λ is set to 0.05, it sits
> *exactly* on top of that true coefficient and thresholds it to zero, so even the clean
> degree-2 baseline "fails" — STLS returns an empty ẏ equation. We therefore use λ = 0.025,
> safely below the smallest true coefficient but above spurious terms. This fragility is itself
> a result: because Lotka's coefficients are an order of magnitude smaller than Lorenz's, the
> usable threshold window is narrow, which connects directly to the threshold-sensitivity
> analysis in the paper (Table 3). A natural follow-up experiment is to sweep λ as a third axis
> alongside degree and noise.

In [ ]:
def lotka_volterra(
    t,
    state,
    alpha=ALPHA,
    beta=BETA,
    delta=DELTA,
    gamma=GAMMA
):
    prey, predator = state

    dprey = alpha*prey - beta*prey*predator
    dpred = delta*prey*predator - gamma*predator

    return [dprey, dpred]

In [ ]:
def generate_lotka_data():

    t_eval = np.arange(
        TSPAN[0],
        TSPAN[1],
        DT
    )

    sol = solve_ivp(
        lotka_volterra,
        TSPAN,
        X0,
        t_eval=t_eval,
        method="RK45",
        rtol=1e-10,
        atol=1e-10
    )

    X = sol.y.T
    t = sol.t

    Xdot = np.zeros_like(X)

    for i in range(len(t)):
        Xdot[i] = lotka_volterra(t[i], X[i])

    return X, Xdot, t

In [ ]:
def build_library(X, degree):

    m, n = X.shape

    Theta = np.ones((m, 1))
    labels = ["1"]

    variables = ["x", "y"][:n]

    for d in range(1, degree+1):

        for combo in combinations_with_replacement(range(n), d):

            col = np.ones(m)
            label = ""

            for idx in combo:
                col *= X[:, idx]
                label += variables[idx]

            Theta = np.column_stack([Theta, col])
            labels.append(label)

    return Theta, labels

In [ ]:
def stls(
    Theta,
    xdot,
    threshold,
    max_iter=100
):

    xi, *_ = np.linalg.lstsq(Theta, xdot, rcond=None)

    for _ in range(max_iter):

        small = np.abs(xi) < threshold
        xi[small] = 0
        active = ~small

        if np.sum(active) == 0:
            break

        xi_active, *_ = np.linalg.lstsq(
            Theta[:, active],
            xdot,
            rcond=None
        )

        xi[active] = xi_active

        if np.array_equal(small, np.abs(xi) < threshold):
            break

    return xi


def run_sindy(
    Theta,
    Xdot,
    threshold=DEFAULT_THRESHOLD
):

    p = Theta.shape[1]
    n_eq = Xdot.shape[1]

    Xi = np.zeros((p, n_eq))

    for k in range(n_eq):
        Xi[:, k] = stls(Theta, Xdot[:, k], threshold)

    return Xi

In [ ]:
def add_noise(X, noise_level):

    if noise_level == 0:
        return X.copy()

    return X + noise_level*np.random.randn(*X.shape)


def finite_diff(X, dt):

    Xdot = np.zeros_like(X)

    Xdot[0] = (X[1]-X[0])/dt
    Xdot[1:-1] = (X[2:] - X[:-2])/(2*dt)
    Xdot[-1] = (X[-1]-X[-2])/dt

    return Xdot

In [ ]:
def get_true_coefficients(labels):
    """
    True Lotka-Volterra coefficients in the polynomial library.

        xdot = alpha*x - beta*x*y      -> x: +ALPHA,  xy: -BETA
        ydot = delta*x*y - gamma*y     -> y: -GAMMA,  xy: +DELTA

    Columns: [xdot, ydot]
    """
    Xi_true = np.zeros((len(labels), 2))

    for i, label in enumerate(labels):

        if label == "x":
            Xi_true[i, 0] = ALPHA      # +1.0
        if label == "xy":
            Xi_true[i, 0] = -BETA      # -0.1

        if label == "y":
            Xi_true[i, 1] = -GAMMA     # -1.5
        if label == "xy":
            Xi_true[i, 1] = DELTA      # +0.05

    return Xi_true


def check_support(Xi_true, Xi_rec, tol=1e-6):

    true_nz = np.abs(Xi_true) > tol
    rec_nz = np.abs(Xi_rec) > tol

    return np.array_equal(true_nz, rec_nz)


def coeff_error(Xi_true, Xi_rec):

    return (
        np.linalg.norm(Xi_true - Xi_rec)
        / np.linalg.norm(Xi_true)
    )

In [ ]:
def mutual_coherence(Theta):

    Theta_norm = Theta / np.linalg.norm(
        Theta,
        axis=0,
        keepdims=True
    )

    G = np.abs(Theta_norm.T @ Theta_norm)

    np.fill_diagonal(G, 0)

    return np.max(G)

In [ ]:
# ==========================================================
# SINGULAR-VALUE DIAGNOSTICS
# ==========================================================
# sigma_min is the smallest singular value of the column-normalised library.
# It measures how close the candidate columns are to being linearly
# dependent (near-rank-deficiency), which is what destabilises the
# least-squares step inside STLS and lets thresholding pick spurious supports.

def singular_value_diagnostics(Theta, rank_tol=None):
    """
    Returns (sigma_min, sigma_max, numerical_rank, n_cols) for the
    column-normalised library Theta.
    """
    norms = np.linalg.norm(Theta, axis=0, keepdims=True)
    norms[norms == 0] = 1.0
    Theta_n = Theta / norms

    s = np.linalg.svd(Theta_n, compute_uv=False)

    sigma_max = s[0]
    sigma_min = s[-1]

    if rank_tol is None:
        rank_tol = sigma_max * max(Theta_n.shape) * np.finfo(float).eps

    numerical_rank = int(np.sum(s > rank_tol))

    return sigma_min, sigma_max, numerical_rank, Theta_n.shape[1]

> **Runtime knob.** The diagnostics (SVD-based) barely vary across seeds, so they are
> computed on only the first `n_diag_seeds` (default 5) while STLS recovery runs on all
> 100 seeds. Recovery probability is identical; the sweep is much faster. Use
> `n_diag_seeds=100` for the original exhaustive behaviour.

In [ ]:
def recovery_probability_experiment(
    X,
    Xdot,
    dt,
    degrees,
    noise_levels,
    n_seeds=100,
    n_diag_seeds=5,
    threshold=DEFAULT_THRESHOLD
):
    """
    Support recovery probability across random seeds for Lotka-Volterra.

    SPEED NOTE: the SVD-based diagnostics (condition number, coherence, sigma_min)
    barely vary across seeds for a fixed (degree, noise), so we compute them on only
    the first `n_diag_seeds` seeds while running the cheap STLS recovery on all
    `n_seeds`. Recovery probability is unchanged. Set n_diag_seeds = n_seeds for the
    original exhaustive behaviour.

    Returns a DataFrame with, per (degree, noise):
        degree, noise, library_size, recovery_probability,
        mean_condition / std_condition, mean_coherence / std_coherence,
        mean_sigma_min / std_sigma_min, mean_sigma_max,
        mean_numerical_rank, mean_rank_deficiency, n_diag_seeds
    """

    results = []

    for deg in degrees:

        print(f"Degree {deg}")

        for noise in noise_levels:

            successes = 0

            cond_numbers = []
            coherences = []
            sigma_mins = []
            sigma_maxs = []
            ranks = []

            for seed in range(n_seeds):

                np.random.seed(seed)

                X_noisy = add_noise(X, noise)

                if noise == 0:
                    Xdot_used = Xdot
                else:
                    Xdot_used = finite_diff(X_noisy, dt)

                Theta, labels = build_library(X_noisy, deg)

                if seed < n_diag_seeds:
                    cond_numbers.append(np.linalg.cond(Theta))
                    coherences.append(mutual_coherence(Theta))
                    s_min, s_max, n_rank, n_cols = singular_value_diagnostics(Theta)
                    sigma_mins.append(s_min)
                    sigma_maxs.append(s_max)
                    ranks.append(n_rank)

                Xi_rec = run_sindy(Theta, Xdot_used, threshold)
                Xi_true = get_true_coefficients(labels)

                if check_support(Xi_true, Xi_rec):
                    successes += 1

            library_size = Theta.shape[1]
            mean_rank = np.mean(ranks)

            results.append({
                "degree": deg,
                "noise": noise,
                "library_size": library_size,

                "recovery_probability": successes / n_seeds,

                "mean_condition": np.mean(cond_numbers),
                "std_condition": np.std(cond_numbers),

                "mean_coherence": np.mean(coherences),
                "std_coherence": np.std(coherences),

                "mean_sigma_min": np.mean(sigma_mins),
                "std_sigma_min": np.std(sigma_mins),
                "mean_sigma_max": np.mean(sigma_maxs),
                "mean_numerical_rank": mean_rank,
                "mean_rank_deficiency": library_size - mean_rank,

                "n_diag_seeds": n_diag_seeds,
            })

    return pd.DataFrame(results)

In [ ]:
# Quick sanity check: clean degree-2 recovery should be exact.
X, Xdot, t = generate_lotka_data()

Theta2, labels2 = build_library(X, 2)
Xi = run_sindy(Theta2, Xdot, DEFAULT_THRESHOLD)
Xi_true = get_true_coefficients(labels2)

print(f"{'term':<8}{'xdot_rec':>10}{'xdot_true':>11}{'ydot_rec':>11}{'ydot_true':>11}")
for i, lab in enumerate(labels2):
    print(f"{lab:<8}{Xi[i,0]:>10.4f}{Xi_true[i,0]:>11.4f}{Xi[i,1]:>11.4f}{Xi_true[i,1]:>11.4f}")

print("\nSupport correct:", check_support(Xi_true, Xi))
print("Coeff error:", coeff_error(Xi_true, Xi))

In [ ]:
df = recovery_probability_experiment(
    X=X,
    Xdot=Xdot,
    dt=DT,
    degrees=DEGREES,
    noise_levels=NOISE_LEVELS,
    n_seeds=100,
    threshold=DEFAULT_THRESHOLD
)

df

In [ ]:
df.to_csv("lotka_recovery_results.csv", index=False)

---
# Visualisations

A clean, consistent figure suite mirroring the Lorenz notebook. Run the **style**
cell first, then each plot. Groups: (1) the system itself, (2) headline recovery
heatmap, (3) recovery vs degree and vs noise, (4) conditioning diagnostics,
(5) mechanistic recovery-vs-diagnostic scatters, (6) the degree-5 anomaly.

In [ ]:
# ==========================================================
# PLOTTING STYLE  (run once; applies to every figure below)
# ==========================================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "savefig.bbox": "tight",
    "font.size": 11, "font.family": "DejaVu Sans",
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 12,
    "axes.labelsize": 11, "axes.labelpad": 6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.9, "axes.edgecolor": "#444444",
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6, "grid.color": "#888888",
    "legend.frameon": False, "legend.fontsize": 9,
    "xtick.color": "#333333", "ytick.color": "#333333",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 2.0, "lines.markersize": 6,
})

SYSTEM_NAME = "Lotka–Volterra"
RECOV_CMAP = LinearSegmentedColormap.from_list("recov", ["#d73027", "#fee08b", "#1a9850"])

def _noise_label(nl):
    return "η = 0" if nl == 0 else f"η = {nl:g}"

def noise_colors(noise_levels):
    cmap = plt.cm.viridis
    n = len(noise_levels)
    return {nl: cmap(0.08 + 0.84*i/(max(n-1,1))) for i, nl in enumerate(sorted(noise_levels))}

def degree_colors(degrees):
    cmap = plt.cm.plasma
    n = len(degrees)
    return {d: cmap(0.05 + 0.85*i/(max(n-1,1))) for i, d in enumerate(sorted(degrees))}


### 1 · The system: time series and phase portrait

In [ ]:
# ---- The system itself: time series + phase portrait ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(t, X[:,0], color="#2166ac", label="prey  x")
axes[0].plot(t, X[:,1], color="#b2182b", label="predator  y")
axes[0].set_xlabel("time"); axes[0].set_ylabel("population")
axes[0].set_title("Lotka–Volterra time series"); axes[0].legend()
axes[1].plot(X[:,0], X[:,1], color="#5aae61", lw=1.6)
axes[1].set_xlabel("prey  x"); axes[1].set_ylabel("predator  y")
axes[1].set_title("phase portrait (closed orbit)")
fig.tight_layout(); plt.show()


### 2 · Headline: recovery probability heatmap

In [ ]:
# ---- Headline plot: recovery probability heatmap (degree x noise) ----
import numpy as np
degs = sorted(df.degree.unique()); noises = sorted(df.noise.unique())
M = np.full((len(noises), len(degs)), np.nan)
for i, nl in enumerate(noises):
    for j, d in enumerate(degs):
        v = df[(df.noise == nl) & (df.degree == d)].recovery_probability
        if len(v): M[i, j] = v.values[0]

fig, ax = plt.subplots(figsize=(8.6, 5.4))
im = ax.imshow(M, cmap=RECOV_CMAP, vmin=0, vmax=1, aspect="auto", origin="upper")
ax.set_xticks(range(len(degs))); ax.set_xticklabels([f"deg {d}" for d in degs])
ax.set_yticks(range(len(noises))); ax.set_yticklabels([_noise_label(n) for n in noises])
for i in range(len(noises)):
    for j in range(len(degs)):
        val = M[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if (val < 0.28 or val > 0.72) else "#222222",
                    fontsize=9, fontweight="bold")
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Noise level η")
ax.set_title(f"{SYSTEM_NAME}: Support Recovery Probability")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("Recovery probability")
ax.grid(False)
fig.tight_layout(); plt.show()


### 3 · Recovery probability vs degree and vs noise

In [ ]:
# ---- Recovery probability vs degree (one line per noise) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.plot(s.degree, 100*s.recovery_probability, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Recovery probability (%)")
ax.set_title(f"{SYSTEM_NAME}: Recovery Probability vs Library Degree")
ax.set_ylim(-4, 104)
ax.legend(title="Noise", ncol=2, loc="lower left")
fig.tight_layout(); plt.show()


In [ ]:
# ---- Recovery probability vs noise (one line per degree): the ANOMALY view ----
# eta=0 is plotted at the far left on a log axis. A line that JUMPS UP from
# eta=0 to the first nonzero noise is the "noise breaks a degeneracy" anomaly.
dc = degree_colors(df.degree.unique())
pos_min = df[df.noise > 0].noise.min()
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for d in sorted(df.degree.unique()):
    s = df[df.degree == d].sort_values("noise").copy()
    s["xn"] = s.noise.replace(0, pos_min/3)
    ax.plot(s.xn, 100*s.recovery_probability, marker="o", color=dc[d],
            label=f"deg {d}", alpha=0.9)
ax.set_xscale("log")
ax.set_xlabel("Noise level η   (η = 0 shown at far left)")
ax.set_ylabel("Recovery probability (%)")
ax.set_title(f"{SYSTEM_NAME}: Recovery Probability vs Noise  (anomaly view)")
ax.set_ylim(-4, 108)
ax.legend(title="Degree", ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout(); plt.show()


### 4 · Library-conditioning diagnostics vs degree

In [ ]:
# ---- Condition number vs degree (log scale, one line per noise) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.semilogy(s.degree, s.mean_condition, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Condition number κ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Library Conditioning vs Degree")
ax.legend(title="Noise", ncol=2)
fig.tight_layout(); plt.show()


In [ ]:
# ---- Condition number growth law: log10(kappa) vs degree, linear fit (eta=0) ----
import numpy as np
s = df[df.noise == 0].sort_values("degree")
d = s.degree.values.astype(float); logk = np.log10(s.mean_condition.values)
slope, intercept = np.polyfit(d, logk, 1)
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.plot(d, logk, "o", color="#2166ac", markersize=8, label="measured (η=0)")
dd = np.linspace(d.min(), d.max(), 100)
ax.plot(dd, slope*dd + intercept, "--", color="#b2182b",
        label=f"fit:  log₁₀κ ≈ {slope:.2f}·d + {intercept:.2f}")
ax.set_xlabel("Polynomial library degree d"); ax.set_ylabel("log₁₀ κ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Condition Number Growth Law")
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()
print(f"Growth law: kappa ~ 10^({slope:.2f} d), i.e. multiply by ~{10**slope:.0f}x per degree")


On Lorenz, coherence saturated near 1 and barely discriminated between degrees. Worth checking whether Lotka is any different — plot it and compare the spread in the recovery-vs-coherence scatter below against recovery-vs-κ and recovery-vs-σ_min.

In [ ]:
# ---- Mutual coherence vs degree ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.plot(s.degree, s.mean_coherence, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("Mutual coherence μ(Θ)")
ax.set_title(f"{SYSTEM_NAME}: Mutual Coherence vs Degree")
ax.legend(title="Noise", ncol=2, loc="lower right")
fig.tight_layout(); plt.show()


In [ ]:
# ---- Smallest singular value sigma_min vs degree (log scale) ----
nc = noise_colors(df.noise.unique())
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for nl in sorted(df.noise.unique()):
    s = df[df.noise == nl].sort_values("degree")
    ax.semilogy(s.degree, s.mean_sigma_min, marker="o", color=nc[nl], label=_noise_label(nl))
ax.set_xlabel("Polynomial library degree"); ax.set_ylabel("σ_min  (column-normalised Θ)")
ax.set_title(f"{SYSTEM_NAME}: Onset of Near-Rank-Deficiency vs Degree")
ax.legend(title="Noise", ncol=2)
fig.tight_layout(); plt.show()


In [ ]:
# ---- Three library diagnostics side by side (eta=0) ----
s = df[df.noise == 0].sort_values("degree")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].semilogy(s.degree, s.mean_condition, "o-", color="#2166ac")
axes[0].set_title("Condition number κ"); axes[0].set_xlabel("degree"); axes[0].set_ylabel("κ(Θ)")
axes[1].plot(s.degree, s.mean_coherence, "o-", color="#5aae61")
axes[1].set_title("Mutual coherence μ"); axes[1].set_xlabel("degree"); axes[1].set_ylabel("μ(Θ)")
axes[2].semilogy(s.degree, s.mean_sigma_min, "o-", color="#b2182b")
axes[2].set_title("Smallest singular value σ_min"); axes[2].set_xlabel("degree"); axes[2].set_ylabel("σ_min")
fig.suptitle(f"{SYSTEM_NAME}: Library Diagnostics vs Degree (η=0)", fontsize=14, fontweight="bold", y=1.04)
fig.tight_layout(); plt.show()


### 5 · Mechanistic links: recovery vs each diagnostic

In [ ]:
# ---- Recovery probability vs each diagnostic (the mechanistic links) ----
# If recovery collapses onto a clean curve against one diagnostic, that
# diagnostic is the better predictor. Compare the spread on each x-axis.
degs = sorted(df.degree.unique())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
specs = [("mean_condition", "Condition number κ", True),
         ("mean_sigma_min", "σ_min", True),
         ("mean_coherence", "Mutual coherence μ", False)]
for ax, (col, lab, logx) in zip(axes, specs):
    sc = ax.scatter(df[col], 100*df.recovery_probability, c=df.degree, cmap="plasma",
                    s=55, edgecolor="white", linewidth=0.5, vmin=min(degs), vmax=max(degs))
    if logx: ax.set_xscale("log")
    ax.set_xlabel(lab); ax.set_ylabel("Recovery probability (%)"); ax.set_ylim(-4, 104)
    ax.set_title(f"vs {lab}")
cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02); cb.set_label("degree")
fig.suptitle(f"{SYSTEM_NAME}: Recovery Probability vs Library Diagnostics",
             fontsize=14, fontweight="bold", y=1.04)
plt.show()


### 6 · The degree-5 anomaly, visualised

Lotka reproduces the Lorenz degree-6 phenomenon one degree earlier (at degree 5):
recovery is 0 at η=0 where σ_min ≈ 10⁻¹¹, then jumps up under a whisker of noise.
The trace below follows the smallest true coefficient — the `xy` term in ẏ (δ=0.05) —
as noise lifts the near-degeneracy.

In [ ]:
# ---- Degree-5 anomaly: ydot-equation threshold-crossing plot ----
# Lotka reproduces the Lorenz degree-6 story at degree 5: recovery is 0 at eta=0
# (sigma_min is at ~1e-11, deeply degenerate) but jumps up under tiny noise.
# Here we trace the smallest true coefficient -- the xy term in ydot (true 0.05) --
# and a spurious competitor, as noise increases.
deg_anom = 5
THR = DEFAULT_THRESHOLD
noise_grid = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

xy_mag, comp_mag = [], []
for nl in noise_grid:
    np.random.seed(0)
    if nl == 0:
        Xn, Xd = X, Xdot
    else:
        Xn = add_noise(X, nl); Xd = finite_diff(Xn, DT)
    Theta_a, labels_a = build_library(Xn, deg_anom)
    xi_y, *_ = np.linalg.lstsq(Theta_a, Xd[:, 1], rcond=None)  # ydot equation
    xy_mag.append(abs(xi_y[labels_a.index("xy")]))
    # a representative spurious higher-degree competitor correlated with xy
    comp_label = "xyy" if "xyy" in labels_a else labels_a[-1]
    comp_mag.append(abs(xi_y[labels_a.index(comp_label)]))

pos_min = min(n for n in noise_grid if n > 0)
xgrid = [n if n > 0 else pos_min/3 for n in noise_grid]

fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.plot(xgrid, xy_mag, "o-", color="#1a9850", label="|coeff(xy)| in ẏ — true term (δ=0.05)")
ax.plot(xgrid, comp_mag, "s-", color="#762a83", label=f"|coeff({comp_label})| — spurious competitor")
ax.axhline(THR, ls="--", color="#b2182b", lw=1.6, label=f"STLS threshold λ = {THR}")
ax.fill_between(xgrid, 0, THR, color="#b2182b", alpha=0.06)
ax.set_xscale("log")
ax.set_xlabel("Noise level η   (η = 0 shown at far left)")
ax.set_ylabel("Initial least-squares coefficient magnitude")
ax.set_title(f"{SYSTEM_NAME} Degree-5 Anomaly: ẏ-equation threshold crossing")
# Focus the y-axis on the threshold region: at high noise the corrupted
# finite-difference derivatives inflate coefficients far past the threshold,
# which is not the part of the story we care about here.
ax.set_ylim(-0.02, max(THR*8, 0.4))
ax.legend(loc="upper left")
fig.tight_layout(); plt.show()
